# Classificação do Dataset Adult com Fuzzy C-Means

Este notebook implementa a classificação do dataset Adult utilizando Fuzzy C-Means supervisionado, com download automático dos dados, avaliação em 30 execuções e salvamento dos resultados (matriz de confusão e gráficos) na pasta `img`.

In [ ]:
# Importar bibliotecas necessárias
import os
import numpy as np
import pandas as pd
from urllib.request import urlretrieve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns
import skfuzzy as fuzz

In [ ]:
# Carregar e visualizar o dataset Adult a partir da pasta data
import pandas as pd

columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
    'hours-per-week', 'native-country', 'income'
]
data_path = 'data/adult.data'
test_path = 'data/adult.test'
df_train = pd.read_csv(data_path, names=columns, sep=',', skipinitialspace=True)
df_test = pd.read_csv(test_path, names=columns, sep=',', skipinitialspace=True, skiprows=1)
df = pd.concat([df_train, df_test], ignore_index=True)
df = df.replace('?', pd.NA).dropna()
df.head()

In [ ]:
# Pré-processamento dos dados
from sklearn.preprocessing import LabelEncoder

def preprocess_adult(df):
    X = df.drop('income', axis=1)
    y = df['income'].apply(lambda x: 1 if '>50K' in x else 0)
    X = pd.get_dummies(X)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, y.values

X, y = preprocess_adult(df)
print(f"Shape dos dados após preprocessamento: {X.shape}")

In [ ]:
# Implementação do classificador Fuzzy C-Means supervisionado

def fuzzy_c_means_predict(X_train, y_train, X_test, n_clusters):
    cntr, u, _, _, _, _, _ = fuzz.cluster.cmeans(
        X_train.T, c=n_clusters, m=2, error=0.005, maxiter=1000, seed=42)
    cluster_labels = np.zeros(n_clusters, dtype=int)
    cluster_membership = np.argmax(u, axis=0)
    for i in range(n_clusters):
        mask = (cluster_membership == i)
        if np.any(mask):
            cluster_labels[i] = mode(y_train[mask], keepdims=True)[0][0]
        else:
            cluster_labels[i] = 0
    u_test, _, _, _, _, _ = fuzz.cluster.cmeans_predict(
        X_test.T, cntr, m=2, error=0.005, maxiter=1000)
    clusters_pred = np.argmax(u_test, axis=0)
    y_pred = cluster_labels[clusters_pred]
    return y_pred

In [ ]:
# Treinamento, avaliação e salvamento dos resultados
accs = []
cms = []
for seed in range(1, 31):
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=seed, stratify=y)
    n_clusters = len(np.unique(y_train))
    y_pred = fuzzy_c_means_predict(X_train, y_train, X_test, n_clusters)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    accs.append(acc)
    cms.append(cm)
    print(f"Execução {seed:02d}: Acurácia = {acc:.4f}")
accs = np.array(accs)
cms = np.array(cms)

# Salvar matrizes e gráficos
np.save('Trabalho03/img/fuzzy_adult_matrizes_confusao.npy', cms)
np.save('Trabalho03/img/fuzzy_adult_acuracias.npy', accs)

# Gráfico de acurácia
plt.figure(figsize=(8,4))
plt.plot(range(1,31), accs, marker='o')
plt.title('Acurácia em cada execução (Fuzzy C-Means + Adult)')
plt.xlabel('Execução')
plt.ylabel('Acurácia')
plt.grid()
plt.savefig('Trabalho03/img/fuzzy_adult_acuracia.png')
plt.show()

# Matriz de confusão média
cm_mean = np.round(cms.mean(axis=0)).astype(int)
sns.heatmap(cm_mean, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusão Média (Fuzzy C-Means + Adult)')
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.savefig('Trabalho03/img/fuzzy_adult_cm_media.png')
plt.show()

print(f"Acurácia média: {accs.mean():.4f}")
print(f"Desvio padrão da acurácia: {accs.std():.4f}")